In [1]:
# C:/Plegma_Programming/src/train_final_api_models_rf_plegma_final.py
# ============================================================
# FINAL API RANDOM FOREST MODELS - PLEGMA GENERIC UI SCRIPT
#
# Corrected final RF version for the PLEGMA API/UI.
# The application must NOT depend on user-provided home_id.
#
# Trains:
#   1) RF with_history_generic
#      - extended lag / rolling features
#      - NO home_id feature
#      - NO train-only home statistics
#      - NO consumption_regime feature/calibrator
#      - home_id is used only internally for creating lags and evaluation grouping
#
#   2) RF no_history
#      - static + time + weather/environmental features
#      - NO home_id
#      - NO lag/history features
#      - final refit on all data before test
#
# Expected input:
#   C:/Plegma_Programming/PLEGMA_final_hourly_dataset.csv
#
# Output:
#   C:/Plegma_Programming/processed/models/final_api_models/RF/
#     ├── training_summary.json
#     ├── with_history/
#     │   ├── model.joblib
#     │   ├── preprocessor.pkl
#     │   ├── feature_config.json
#     │   ├── metadata.json
#     │   └── test_predictions.csv
#     └── no_history/
#         ├── model.joblib
#         ├── preprocessor.pkl
#         ├── feature_config.json
#         ├── metadata.json
#         ├── test_predictions.csv
#         ├── nohistory_test_comparison.csv
#         └── metrics_by_home.csv
# ============================================================

from __future__ import annotations

import json
import math
import shutil
import time
import warnings
from pathlib import Path
from typing import Any, Dict, Tuple

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

BASE_DIR = Path(r"C:/Plegma_Programming")
DATA_PATH = BASE_DIR / "PLEGMA_final_hourly_dataset.csv"

OUT_ROOT = BASE_DIR / "processed" / "models" / "final_api_models" / "RF2"
WITH_HISTORY_DIR = OUT_ROOT / "with_history"
NO_HISTORY_DIR = OUT_ROOT / "no_history"

OUT_ROOT.mkdir(parents=True, exist_ok=True)

TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

RANDOM_STATE = 42

TEST_DAYS = 30
VAL_DAYS_TOTAL = 30
CAL_DAYS = 15

# PLEGMA houses that were excluded earlier because of extensive missing data.
# If they are already absent from PLEGMA_final_hourly_dataset.csv, nothing happens.
BAD_HOMES = ["House_06", "House_08", "House_12", "house06", "house08", "house12", "06", "08", "12"]
REMOVE_BAD_HOMES = True

TRAIN_WITH_HISTORY = True
TRAIN_NO_HISTORY = True

CLEAR_WITH_HISTORY_DIR = True
CLEAR_NO_HISTORY_DIR = True

USE_LOG_TARGET = True

# Same idea as IDEAL: no-history keeps clipping, with-history avoids clipping by default.
CLIP_TARGET_TRAIN_NO_HISTORY = True
CLIP_TARGET_TRAIN_HISTORY = False
CLIP_Q_LOW = 0.005
CLIP_Q_HIGH = 0.995

UNKNOWN_LABEL = "unknown"

# PLEGMA numeric/static/environmental columns.
# Missing columns are created as NaN and handled by the preprocessor.
PLEGMA_NUMERIC_BASE = [
    "external_temperature",
    "internal_temperature",
    "external_humidity",
    "internal_humidity",
    "num_rooms",
    "residents",
    "num_adults",
    "num_children",
    "num_elderly",
    "has_ac",
    "has_fridge_freezer",
    "has_dryer",
    "has_washing_machine",
    "has_dishwasher",
    "has_microwave",
    "has_electric_oven",
    "has_electric_hob",
    "solar_panels",
]

TIME_NUMERIC_EXTENDED = [
    "hour",
    "day_of_week",
    "month",
    "day_of_month",
    "week_of_year",
    "is_weekend",
    "is_holiday",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos",
]

# no_history: no home_id and no lag/history features.
NO_HISTORY_NUMERIC = [
    "hour",
    "day_of_week",
    "is_weekend",
    "is_holiday",
    "month",
    "external_temperature",
    "internal_temperature",
    "external_humidity",
    "internal_humidity",
    "num_rooms",
    "residents",
    "num_adults",
    "num_children",
    "num_elderly",
    "has_ac",
    "has_fridge_freezer",
    "has_dryer",
    "has_washing_machine",
    "has_dishwasher",
    "has_microwave",
    "has_electric_oven",
    "has_electric_hob",
    "solar_panels",
]

NO_HISTORY_CATEGORICAL = [
    "building_type",
    "build_era",
    "season",
    "income_band",
    "heating_type",
    "water_heater_type",
    "homeowner_status",
    "years_in_house",
    "occupation",
]

# with_history generic: no home_id, no home_stats, no consumption_regime.
# home_id is used only internally for lag construction/evaluation, never as a model feature.
HISTORY_CATEGORICAL = [
    "building_type",
    "build_era",
    "season",
    "income_band",
    "heating_type",
    "water_heater_type",
    "homeowner_status",
    "years_in_house",
    "occupation",
]

LAG_HOURS = [1, 2, 3, 6, 12, 24, 48, 72, 168, 336]
ROLLING_WINDOWS = [3, 6, 12, 24, 48, 168]

REQUIRED_HISTORY_COLS = [
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "lag_336h",
    "roll_mean_24h",
    "roll_mean_168h",
]

# PLEGMA is much smaller than IDEAL, so by default no sampling is needed.
# Set e.g. 350_000 if the file becomes large.
MAX_RF_TRAIN_ROWS_WITH_HISTORY = None
MAX_RF_TRAIN_ROWS_NO_HISTORY = None

# RF parameters.
# with_history now uses the same generic policy as LGBM/XGB: no home_id-dependent features.
# Starting point: best balanced squared-error RF candidate from the previous search, retrained generically.
RF_WITH_HISTORY_CANDIDATE_NAME = "wh_more_trees_leaf10_depth24_generic_no_home_id"
RF_WITH_HISTORY_PARAMS = dict(
    n_estimators=500,
    criterion="squared_error",
    max_depth=24,
    min_samples_leaf=10,
    min_samples_split=20,
    max_features=0.70,
    bootstrap=True,
    max_samples=0.85,
    n_jobs=-1,
    random_state=RANDOM_STATE + 12,
    verbose=1,
)

# no_history selected: nh_regularized_leaf20_depth18
# Rationale: better RMSE/daily error than the previous RF no_history while keeping MAE almost unchanged.
RF_NO_HISTORY_CANDIDATE_NAME = "nh_regularized_leaf20_depth18"
RF_NO_HISTORY_PARAMS = dict(
    n_estimators=500,
    criterion="squared_error",
    max_depth=18,
    min_samples_leaf=20,
    min_samples_split=40,
    max_features=0.55,
    bootstrap=True,
    n_jobs=-1,
    random_state=RANDOM_STATE + 24,
    verbose=1,
)


# ============================================================
# UTILS
# ============================================================

def save_json(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def file_size_mb(path: Path) -> float:
    if not path.exists():
        return float("nan")
    return path.stat().st_size / (1024 * 1024)


def mape_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mask = y_true > eps
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def smape_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100)


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(math.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE_%": mape_safe(y_true, y_pred),
        "sMAPE_%": smape_safe(y_true, y_pred),
        "bias_Wh": float(np.mean(y_pred - y_true)),
    }


def evaluate_predictions(df_eval: pd.DataFrame, pred_col: str) -> Dict[str, float]:
    y_true = df_eval[TARGET_COL].values
    y_pred = df_eval[pred_col].values
    metrics = compute_metrics(y_true, y_pred)

    daily = (
        df_eval
        .assign(date=df_eval[TIME_COL].dt.date)
        .groupby([HOME_COL, "date"], as_index=False)
        .agg(
            actual_kWh=(TARGET_COL, lambda x: x.sum() / 1000),
            pred_kWh=(pred_col, lambda x: x.sum() / 1000),
        )
    )
    daily["abs_error_kWh"] = np.abs(daily["actual_kWh"] - daily["pred_kWh"])
    daily["abs_error_pct"] = daily["abs_error_kWh"] / daily["actual_kWh"].replace(0, np.nan) * 100

    metrics.update({
        "daily_abs_error_kWh_mean": float(daily["abs_error_kWh"].mean()),
        "daily_abs_error_pct_mean": float(daily["abs_error_pct"].mean()),
        "num_home_days": int(len(daily)),
    })
    return metrics


def metrics_by_home(df_eval: pd.DataFrame, pred_col: str) -> pd.DataFrame:
    rows = []
    for home, g in df_eval.groupby(HOME_COL):
        m = evaluate_predictions(g, pred_col)
        rows.append({HOME_COL: home, **m})
    return pd.DataFrame(rows).sort_values("RMSE")


def print_block(title: str) -> None:
    print("=" * 100)
    print(title)
    print("=" * 100)


def print_metrics(title: str, metrics: Dict[str, float]) -> None:
    print_block(title)
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")


def transform_target(y):
    y = np.asarray(y, dtype=np.float64)
    if USE_LOG_TARGET:
        return np.log1p(np.maximum(y, 0.0))
    return y


def inverse_target(y_hat):
    y_hat = np.asarray(y_hat, dtype=np.float64)
    if USE_LOG_TARGET:
        return np.expm1(y_hat)
    return y_hat


def maybe_clip_target(y_train, enabled: bool):
    if not enabled:
        return y_train, None
    lo = np.quantile(y_train, CLIP_Q_LOW)
    hi = np.quantile(y_train, CLIP_Q_HIGH)
    return np.clip(y_train, lo, hi), {"low": float(lo), "high": float(hi)}


def sample_train_for_rf(train_df: pd.DataFrame, max_rows, random_state: int) -> pd.DataFrame:
    if max_rows is None or len(train_df) <= max_rows:
        return train_df.copy().reset_index(drop=True)

    frac = max_rows / len(train_df)
    sampled_parts = []
    for _, g in train_df.groupby(HOME_COL, sort=False):
        n = max(1, int(round(len(g) * frac)))
        n = min(n, len(g))
        sampled_parts.append(g.sample(n=n, random_state=random_state))

    sampled = pd.concat(sampled_parts, axis=0)
    if len(sampled) > max_rows:
        sampled = sampled.sample(n=max_rows, random_state=random_state)
    return sampled.sample(frac=1.0, random_state=random_state).reset_index(drop=True)


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def build_preprocessor(numeric_cols, categorical_cols):
    numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=UNKNOWN_LABEL)),
            ("onehot", make_one_hot_encoder()),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


def benchmark_prediction_speed(model, X, sample_sizes=(24, 168, 720)) -> Dict[str, float]:
    results = {}
    n_available = X.shape[0]
    for n in sample_sizes:
        n_use = min(n, n_available)
        if n_use <= 0:
            continue
        X_sample = X[:n_use]
        start = time.perf_counter()
        _ = model.predict(X_sample)
        elapsed = time.perf_counter() - start
        results[f"predict_seconds_{n_use}_rows"] = float(elapsed)
    return results


# ============================================================
# COMMON FEATURE ENGINEERING
# ============================================================

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["hour"] = df[TIME_COL].dt.hour
    df["day_of_week"] = df[TIME_COL].dt.dayofweek
    df["month"] = df[TIME_COL].dt.month
    df["day_of_month"] = df[TIME_COL].dt.day
    df["week_of_year"] = df[TIME_COL].dt.isocalendar().week.astype(int)
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

    if "is_holiday" not in df.columns:
        df["is_holiday"] = 0

    if "season" not in df.columns:
        df["season"] = df["month"].map({
            12: "winter", 1: "winter", 2: "winter",
            3: "spring", 4: "spring", 5: "spring",
            6: "summer", 7: "summer", 8: "summer",
            9: "autumn", 10: "autumn", 11: "autumn",
        }).fillna(UNKNOWN_LABEL)

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df


def temporal_split(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    max_time = df[TIME_COL].max()
    test_start = max_time - pd.Timedelta(days=TEST_DAYS) + pd.Timedelta(hours=1)
    val_start = test_start - pd.Timedelta(days=VAL_DAYS_TOTAL)
    cal_start = test_start - pd.Timedelta(days=CAL_DAYS)

    train_df = df[df[TIME_COL] < val_start].copy()
    es_df = df[(df[TIME_COL] >= val_start) & (df[TIME_COL] < cal_start)].copy()
    cal_df = df[(df[TIME_COL] >= cal_start) & (df[TIME_COL] < test_start)].copy()
    test_df = df[df[TIME_COL] >= test_start].copy()
    return train_df, es_df, cal_df, test_df


def normalize_home_id(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[HOME_COL] = df[HOME_COL].astype(str)
    return df


# ============================================================
# WITH-HISTORY FEATURES
# ============================================================

def add_lag_and_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values([HOME_COL, TIME_COL]).copy()
    g = df.groupby(HOME_COL)[TARGET_COL]

    for lag in LAG_HOURS:
        df[f"lag_{lag}h"] = g.shift(lag)

    shifted = g.shift(1)

    # Important: rolling is applied separately inside each home_id group.
    for w in ROLLING_WINDOWS:
        df[f"roll_mean_{w}h"] = (
            shifted
            .groupby(df[HOME_COL])
            .rolling(w, min_periods=max(2, w // 4))
            .mean()
            .reset_index(level=0, drop=True)
        )

    for w in [24, 48, 168]:
        df[f"roll_std_{w}h"] = (
            shifted.groupby(df[HOME_COL]).rolling(w, min_periods=max(2, w // 4)).std().reset_index(level=0, drop=True)
        )
        df[f"roll_min_{w}h"] = (
            shifted.groupby(df[HOME_COL]).rolling(w, min_periods=max(2, w // 4)).min().reset_index(level=0, drop=True)
        )
        df[f"roll_max_{w}h"] = (
            shifted.groupby(df[HOME_COL]).rolling(w, min_periods=max(2, w // 4)).max().reset_index(level=0, drop=True)
        )

    df["lag_1h_minus_24h"] = df["lag_1h"] - df["lag_24h"]
    df["lag_24h_minus_168h"] = df["lag_24h"] - df["lag_168h"]
    df["lag_48h_minus_168h"] = df["lag_48h"] - df["lag_168h"]
    df["roll_24h_div_168h"] = df["roll_mean_24h"] / (df["roll_mean_168h"] + 1.0)
    df["roll_3h_div_24h"] = df["roll_mean_3h"] / (df["roll_mean_24h"] + 1.0)
    return df


def build_train_only_home_stats(train_df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, float]]:
    tmp = train_df.copy()
    tmp["date"] = tmp[TIME_COL].dt.date

    hourly_stats = (
        tmp.groupby(HOME_COL)
        .agg(
            home_mean_Wh=(TARGET_COL, "mean"),
            home_median_Wh=(TARGET_COL, "median"),
            home_std_Wh=(TARGET_COL, "std"),
            home_min_Wh=(TARGET_COL, "min"),
            home_max_Wh=(TARGET_COL, "max"),
            home_p75_Wh=(TARGET_COL, lambda x: np.percentile(x, 75)),
            home_p90_Wh=(TARGET_COL, lambda x: np.percentile(x, 90)),
            home_p95_Wh=(TARGET_COL, lambda x: np.percentile(x, 95)),
            home_p99_Wh=(TARGET_COL, lambda x: np.percentile(x, 99)),
        )
        .reset_index()
    )

    daily = (
        tmp.groupby([HOME_COL, "date"], as_index=False)
        .agg(daily_kWh=(TARGET_COL, lambda x: x.sum() / 1000))
    )

    daily_stats = (
        daily.groupby(HOME_COL)
        .agg(
            home_daily_mean_kWh=("daily_kWh", "mean"),
            home_daily_median_kWh=("daily_kWh", "median"),
            home_daily_std_kWh=("daily_kWh", "std"),
            home_daily_p75_kWh=("daily_kWh", lambda x: np.percentile(x, 75)),
            home_daily_p90_kWh=("daily_kWh", lambda x: np.percentile(x, 90)),
        )
        .reset_index()
    )

    hourly_profile = (
        tmp.groupby([HOME_COL, "hour"], as_index=False)
        .agg(home_hour_mean_Wh=(TARGET_COL, "mean"))
    )
    hourly_profile_wide = hourly_profile.pivot(index=HOME_COL, columns="hour", values="home_hour_mean_Wh").reset_index()
    hourly_profile_wide.columns = [
        HOME_COL if c == HOME_COL else f"home_hour_{int(c):02d}_mean_Wh"
        for c in hourly_profile_wide.columns
    ]

    home_stats = (
        hourly_stats
        .merge(daily_stats, on=HOME_COL, how="left")
        .merge(hourly_profile_wide, on=HOME_COL, how="left")
    )

    q_low = home_stats["home_daily_mean_kWh"].quantile(0.33)
    q_high = home_stats["home_daily_mean_kWh"].quantile(0.66)
    q_very_high = home_stats["home_daily_mean_kWh"].quantile(0.90)

    def assign_regime(x):
        if pd.isna(x):
            return UNKNOWN_LABEL
        if x <= q_low:
            return "low"
        if x <= q_high:
            return "medium"
        if x <= q_very_high:
            return "high"
        return "high_high"

    home_stats["consumption_regime"] = home_stats["home_daily_mean_kWh"].apply(assign_regime)

    thresholds = {
        "q_low_daily_kWh": float(q_low),
        "q_high_daily_kWh": float(q_high),
        "q_very_high_daily_kWh": float(q_very_high),
    }
    return home_stats, thresholds


def fit_regime_calibrator(cal_eval: pd.DataFrame, pred_col: str) -> Dict[str, Any]:
    params = {}
    for regime, g in cal_eval.groupby("consumption_regime"):
        if len(g) < 100:
            params[regime] = {"type": "none", "scale": 1.0}
            continue

        y_true = g[TARGET_COL].values
        y_pred = g[pred_col].values
        raw_scale = float((np.sum(y_true) + 1e-6) / (np.sum(y_pred) + 1e-6))
        raw_scale = float(np.clip(raw_scale, 0.90, 1.20))

        if regime == "high_high":
            alpha = 0.75
        elif regime in ["high", "medium"]:
            alpha = 0.10
        else:
            alpha = 0.0

        scale = 1.0 + alpha * (raw_scale - 1.0)
        if abs(scale - 1.0) < 1e-9:
            params[regime] = {"type": "none", "scale": 1.0}
        else:
            params[regime] = {"type": "scale", "scale": float(scale), "alpha": alpha, "raw_scale": raw_scale}

    for regime in ["low", "medium", "high", "high_high", UNKNOWN_LABEL]:
        params.setdefault(regime, {"type": "none", "scale": 1.0})

    return {"type": "regime_specific", "regime_params": params}


def apply_regime_calibrator(pred: np.ndarray, regimes: np.ndarray, calibrator: Dict[str, Any]) -> np.ndarray:
    pred = np.asarray(pred).copy()
    regimes = pd.Series(regimes).fillna(UNKNOWN_LABEL).astype(str).replace({"nan": UNKNOWN_LABEL}).values
    calibrated = pred.copy()
    for regime, p in calibrator["regime_params"].items():
        mask = regimes == regime
        if mask.sum() == 0:
            continue
        if p["type"] == "scale":
            calibrated[mask] = pred[mask] * p["scale"]
    return np.clip(calibrated, 0, None)


# ============================================================
# DATA LOADING
# ============================================================

def load_and_prepare_base_data() -> Tuple[pd.DataFrame, list]:
    print_block("Loading PLEGMA dataset")
    df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL], low_memory=False)
    df = normalize_home_id(df)
    df = df.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)

    print(f"Rows loaded: {len(df):,}")
    print(f"Homes loaded: {df[HOME_COL].nunique()}")
    print(f"Date range: {df[TIME_COL].min()} to {df[TIME_COL].max()}")

    existing_bad = []
    if REMOVE_BAD_HOMES:
        existing_bad = [h for h in BAD_HOMES if h in set(df[HOME_COL].unique())]
        if existing_bad:
            print("Removing bad homes:", existing_bad)
            df = df[~df[HOME_COL].isin(existing_bad)].copy()

    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[TARGET_COL]).copy()
    df[TARGET_COL] = df[TARGET_COL].clip(lower=0)

    for col in PLEGMA_NUMERIC_BASE:
        if col not in df.columns:
            df[col] = np.nan

    for col in set(NO_HISTORY_CATEGORICAL + HISTORY_CATEGORICAL):
        if col not in df.columns and col != HOME_COL and col != "consumption_regime":
            df[col] = UNKNOWN_LABEL

    df = add_time_features(df)

    for c in set(NO_HISTORY_CATEGORICAL + HISTORY_CATEGORICAL):
        if c in df.columns:
            df[c] = df[c].fillna(UNKNOWN_LABEL).astype(str).replace({"nan": UNKNOWN_LABEL, "None": UNKNOWN_LABEL})

    for c in PLEGMA_NUMERIC_BASE + TIME_NUMERIC_EXTENDED:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df, existing_bad


# ============================================================
# TRAIN WITH-HISTORY RF
# ============================================================

def train_with_history_model(df_base: pd.DataFrame) -> Dict[str, Any]:
    print_block("WITH_HISTORY GENERIC TRAINING - FINAL PLEGMA RF")

    if CLEAR_WITH_HISTORY_DIR and WITH_HISTORY_DIR.exists():
        print("Clearing:", WITH_HISTORY_DIR)
        shutil.rmtree(WITH_HISTORY_DIR)
    WITH_HISTORY_DIR.mkdir(parents=True, exist_ok=True)

    # home_id is used only here to construct leakage-free lags/rolling features per original house.
    # It is NOT included as a model feature.
    df_hist = add_lag_and_rolling_features(df_base)
    available_required = [c for c in REQUIRED_HISTORY_COLS if c in df_hist.columns]
    df_hist = df_hist.dropna(subset=available_required).copy()
    df_hist = df_hist.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)

    print(f"Rows after history features: {len(df_hist):,}")
    print(f"Homes after history features: {df_hist[HOME_COL].nunique()}")

    hist_train, hist_es, hist_cal, hist_test = temporal_split(df_hist)

    print_block("With_history generic temporal split")
    print(f"Train: {hist_train[TIME_COL].min()} to {hist_train[TIME_COL].max()} | rows: {len(hist_train):,}")
    print(f"Early-stop val: {hist_es[TIME_COL].min()} to {hist_es[TIME_COL].max()} | rows: {len(hist_es):,}")
    print(f"Calibration val: {hist_cal[TIME_COL].min()} to {hist_cal[TIME_COL].max()} | rows: {len(hist_cal):,}")
    print(f"Test: {hist_test[TIME_COL].min()} to {hist_test[TIME_COL].max()} | rows: {len(hist_test):,}")

    lag_cols = [c for c in df_hist.columns if c.startswith("lag_")]
    rolling_cols = [c for c in df_hist.columns if c.startswith("roll_")]

    numeric_cols = [
        c for c in PLEGMA_NUMERIC_BASE + TIME_NUMERIC_EXTENDED + lag_cols + rolling_cols
        if c in hist_train.columns
    ]
    categorical_cols = [c for c in HISTORY_CATEGORICAL if c in hist_train.columns]

    # Hard safety checks: no home_id-dependent feature should enter the model.
    forbidden_features = {HOME_COL, "consumption_regime"}
    forbidden_features.update({
        "home_mean_Wh", "home_median_Wh", "home_std_Wh", "home_min_Wh", "home_max_Wh",
        "home_p75_Wh", "home_p90_Wh", "home_p95_Wh", "home_p99_Wh",
        "home_daily_mean_kWh", "home_daily_median_kWh", "home_daily_std_kWh",
        "home_daily_p75_kWh", "home_daily_p90_kWh",
    })
    forbidden_in_model = sorted(forbidden_features.intersection(set(numeric_cols + categorical_cols)))
    if forbidden_in_model:
        raise RuntimeError(f"Forbidden home_id-dependent features entered RF with_history model: {forbidden_in_model}")

    for split_df in [hist_train, hist_es, hist_cal, hist_test]:
        for c in numeric_cols:
            split_df[c] = pd.to_numeric(split_df[c], errors="coerce")
        for c in categorical_cols:
            split_df[c] = split_df[c].fillna(UNKNOWN_LABEL).astype(str).replace({"nan": UNKNOWN_LABEL, "None": UNKNOWN_LABEL})

    print_block("With_history generic RF features")
    print(f"Numeric features: {len(numeric_cols)}")
    print(f"Categorical features: {len(categorical_cols)}")
    print("Categorical:", categorical_cols)
    print("Uses home_id as feature: False")
    print("Uses train-only home stats: False")
    print("Uses consumption_regime: False")

    hist_train_model = sample_train_for_rf(hist_train, MAX_RF_TRAIN_ROWS_WITH_HISTORY, RANDOM_STATE + 101)

    print_block("RF with_history generic training rows")
    print(f"Full train rows: {len(hist_train):,}")
    print(f"Used train rows: {len(hist_train_model):,}")
    print(f"Used homes for training/evaluation grouping only: {hist_train_model[HOME_COL].nunique()}")

    preprocessor = build_preprocessor(numeric_cols, categorical_cols)
    preprocessor.fit(hist_train[numeric_cols + categorical_cols])

    X_train = preprocessor.transform(hist_train_model[numeric_cols + categorical_cols])
    X_cal = preprocessor.transform(hist_cal[numeric_cols + categorical_cols])
    X_test = preprocessor.transform(hist_test[numeric_cols + categorical_cols])

    y_train_raw = hist_train_model[TARGET_COL].values
    y_train_used, clip_bounds = maybe_clip_target(y_train_raw, enabled=CLIP_TARGET_TRAIN_HISTORY)
    y_train = transform_target(y_train_used)

    print_block("With_history generic RF matrices")
    print(f"X_train: {X_train.shape}")
    print(f"X_cal:   {X_cal.shape}")
    print(f"X_test:  {X_test.shape}")

    model = RandomForestRegressor(**RF_WITH_HISTORY_PARAMS)

    print_block("Training WITH_HISTORY GENERIC RF")
    start_train = time.perf_counter()
    model.fit(X_train, y_train)
    train_seconds = float(time.perf_counter() - start_train)

    pred_cal = inverse_target(model.predict(X_cal))
    pred_test = inverse_target(model.predict(X_test))
    pred_cal = np.clip(pred_cal, 0, None)
    pred_test = np.clip(pred_test, 0, None)

    cal_eval = hist_cal[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    cal_eval["prediction_Wh"] = pred_cal

    test_eval = hist_test[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    test_eval["prediction_Wh"] = pred_test

    metrics_cal = evaluate_predictions(cal_eval, "prediction_Wh")
    metrics_test = evaluate_predictions(test_eval, "prediction_Wh")

    print_metrics("DIAGNOSTIC METRICS ON CALIBRATION PERIOD - RF WITH_HISTORY GENERIC", metrics_cal)
    print_metrics("FINAL TEST METRICS - RF WITH_HISTORY GENERIC", metrics_test)

    model_path = WITH_HISTORY_DIR / "model.joblib"
    preprocessor_path = WITH_HISTORY_DIR / "preprocessor.pkl"
    feature_config_path = WITH_HISTORY_DIR / "feature_config.json"
    metadata_path = WITH_HISTORY_DIR / "metadata.json"
    predictions_path = WITH_HISTORY_DIR / "test_predictions.csv"
    metrics_by_home_path = WITH_HISTORY_DIR / "metrics_by_home.csv"

    joblib.dump(model, model_path)
    joblib.dump(preprocessor, preprocessor_path)

    prediction_benchmark = benchmark_prediction_speed(model, X_test)

    feature_config = {
        "mode": "with_history",
        "model_family": "RandomForestRegressor",
        "model_name": "plegma_rf_with_history_generic_no_home_id",
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "target_col": TARGET_COL,
        "time_col": TIME_COL,
        "home_col": HOME_COL,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN_HISTORY,
        "uses_home_id_as_feature": False,
        "requires_user_home_id": False,
        "uses_lag_features": True,
        "uses_rolling_features": True,
        "uses_home_stats": False,
        "uses_consumption_regime": False,
        "uses_regime_calibrator": False,
        "uses_knn_profiles": False,
        "uses_behavior_profiles": False,
        "required_history_cols": available_required,
        "lag_hours": LAG_HOURS,
        "rolling_windows": ROLLING_WINDOWS,
        "default_prediction_column": "prediction_Wh",
        "optional_prediction_column": None,
        "note": "home_id is used only internally during training/evaluation to build lags per original house; it is not a model feature and is not required by the UI/API.",
    }
    save_json(feature_config, feature_config_path)

    metrics_by_home(test_eval, "prediction_Wh").to_csv(metrics_by_home_path, index=False)
    test_eval.to_csv(predictions_path, index=False)

    metadata = {
        "model_family": "RandomForestRegressor",
        "model_name": "plegma_rf_with_history_generic_no_home_id",
        "mode": "with_history",
        "rf_candidate_name": RF_WITH_HISTORY_CANDIDATE_NAME,
        "rf_params": RF_WITH_HISTORY_PARAMS,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN_HISTORY,
        "clip_bounds": clip_bounds,
        "training_seconds": train_seconds,
        "model_file_size_mb": file_size_mb(model_path),
        "preprocessor_file_size_mb": file_size_mb(preprocessor_path),
        "prediction_benchmark": prediction_benchmark,
        "full_train_rows": int(len(hist_train)),
        "used_train_rows": int(len(hist_train_model)),
        "metrics_calibration_diagnostic": metrics_cal,
        "metrics_test": metrics_test,
        "generic_policy": {
            "requires_user_home_id": False,
            "uses_home_id_as_feature": False,
            "uses_home_stats": False,
            "uses_consumption_regime": False,
            "uses_regime_calibrator": False,
            "home_id_usage": "internal_grouping_only_for_training_lag_construction_and_evaluation",
        },
        "artifact_files": {
            "model": str(model_path),
            "preprocessor": str(preprocessor_path),
            "feature_config": str(feature_config_path),
            "metadata": str(metadata_path),
            "predictions": str(predictions_path),
            "metrics_by_home": str(metrics_by_home_path),
        },
        "train_period": {"start": str(hist_train[TIME_COL].min()), "end": str(hist_train[TIME_COL].max()), "rows": int(len(hist_train))},
        "early_stop_validation_period_not_used_by_rf": {"start": str(hist_es[TIME_COL].min()), "end": str(hist_es[TIME_COL].max()), "rows": int(len(hist_es))},
        "calibration_validation_period": {"start": str(hist_cal[TIME_COL].min()), "end": str(hist_cal[TIME_COL].max()), "rows": int(len(hist_cal))},
        "test_period": {"start": str(hist_test[TIME_COL].min()), "end": str(hist_test[TIME_COL].max()), "rows": int(len(hist_test))},
    }
    save_json(metadata, metadata_path)

    return {
        "metrics_calibration_diagnostic": metrics_cal,
        "metrics_test": metrics_test,
        "model_path": str(model_path),
        "preprocessor_path": str(preprocessor_path),
        "feature_config_path": str(feature_config_path),
        "metadata_path": str(metadata_path),
        "predictions_path": str(predictions_path),
        "metrics_by_home_path": str(metrics_by_home_path),
        "training_seconds": train_seconds,
        "model_file_size_mb": file_size_mb(model_path),
        "prediction_benchmark": prediction_benchmark,
    }

# ============================================================
# TRAIN NO-HISTORY RF
# ============================================================

def train_no_history_model(df_base: pd.DataFrame) -> Dict[str, Any]:
    print_block("NO_HISTORY TRAINING - FINAL PLEGMA RF")

    if CLEAR_NO_HISTORY_DIR and NO_HISTORY_DIR.exists():
        print("Clearing:", NO_HISTORY_DIR)
        shutil.rmtree(NO_HISTORY_DIR)
    NO_HISTORY_DIR.mkdir(parents=True, exist_ok=True)

    df_no_history = df_base.sort_values(TIME_COL).reset_index(drop=True).copy()
    train_df, es_df, cal_df, test_df = temporal_split(df_no_history)

    final_train_df = (
        pd.concat([train_df, es_df, cal_df], axis=0)
        .sort_values(TIME_COL)
        .reset_index(drop=True)
    )

    print_block("No_history temporal split")
    print(f"Train: {train_df[TIME_COL].min()} to {train_df[TIME_COL].max()} | rows: {len(train_df):,}")
    print(f"Early-stop val: {es_df[TIME_COL].min()} to {es_df[TIME_COL].max()} | rows: {len(es_df):,}")
    print(f"Calibration val: {cal_df[TIME_COL].min()} to {cal_df[TIME_COL].max()} | rows: {len(cal_df):,}")
    print(f"Final train used: {final_train_df[TIME_COL].min()} to {final_train_df[TIME_COL].max()} | rows: {len(final_train_df):,}")
    print(f"Test: {test_df[TIME_COL].min()} to {test_df[TIME_COL].max()} | rows: {len(test_df):,}")

    numeric_cols = [c for c in NO_HISTORY_NUMERIC if c in final_train_df.columns]
    categorical_cols = [c for c in NO_HISTORY_CATEGORICAL if c in final_train_df.columns]

    for split_df in [final_train_df, train_df, es_df, cal_df, test_df]:
        for c in numeric_cols:
            split_df[c] = pd.to_numeric(split_df[c], errors="coerce")
        for c in categorical_cols:
            split_df[c] = split_df[c].fillna(UNKNOWN_LABEL).astype(str).replace({"nan": UNKNOWN_LABEL, "None": UNKNOWN_LABEL})

    print_block("No_history RF features")
    print(f"Numeric features: {len(numeric_cols)}")
    print(f"Categorical features: {len(categorical_cols)}")
    print("Numeric:", numeric_cols)
    print("Categorical:", categorical_cols)

    train_df_model = sample_train_for_rf(final_train_df, MAX_RF_TRAIN_ROWS_NO_HISTORY, RANDOM_STATE + 202)

    print_block("RF no_history final training rows")
    print(f"Original train rows: {len(train_df):,}")
    print(f"Final train rows including val/cal: {len(final_train_df):,}")
    print(f"Used train rows: {len(train_df_model):,}")
    print(f"Used homes: {train_df_model[HOME_COL].nunique()}")

    preprocessor = build_preprocessor(numeric_cols, categorical_cols)
    preprocessor.fit(final_train_df[numeric_cols + categorical_cols])

    X_train = preprocessor.transform(train_df_model[numeric_cols + categorical_cols])
    X_cal = preprocessor.transform(cal_df[numeric_cols + categorical_cols])
    X_test = preprocessor.transform(test_df[numeric_cols + categorical_cols])

    y_train_raw = train_df_model[TARGET_COL].values
    y_train_used, clip_bounds = maybe_clip_target(y_train_raw, enabled=CLIP_TARGET_TRAIN_NO_HISTORY)
    y_train = transform_target(y_train_used)

    print_block("No_history RF matrices")
    print(f"X_train: {X_train.shape}")
    print(f"X_cal:   {X_cal.shape}")
    print(f"X_test:  {X_test.shape}")

    model = RandomForestRegressor(**RF_NO_HISTORY_PARAMS)

    print_block("Training NO_HISTORY RF")
    start_train = time.perf_counter()
    model.fit(X_train, y_train)
    train_seconds = float(time.perf_counter() - start_train)

    pred_cal = inverse_target(model.predict(X_cal))
    pred_test = inverse_target(model.predict(X_test))
    pred_cal = np.clip(pred_cal, 0, None)
    pred_test = np.clip(pred_test, 0, None)

    cal_eval = cal_df[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    cal_eval["prediction_Wh"] = pred_cal

    test_eval = test_df[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    test_eval["prediction_Wh"] = pred_test

    metrics_cal = evaluate_predictions(cal_eval, "prediction_Wh")
    metrics_test = evaluate_predictions(test_eval, "prediction_Wh")

    print_metrics("DIAGNOSTIC METRICS ON CALIBRATION PERIOD - RF NO_HISTORY", metrics_cal)
    print_metrics("FINAL TEST METRICS - RF NO_HISTORY", metrics_test)

    model_path = NO_HISTORY_DIR / "model.joblib"
    preprocessor_path = NO_HISTORY_DIR / "preprocessor.pkl"
    feature_config_path = NO_HISTORY_DIR / "feature_config.json"
    metadata_path = NO_HISTORY_DIR / "metadata.json"
    predictions_path = NO_HISTORY_DIR / "test_predictions.csv"
    comparison_path = NO_HISTORY_DIR / "nohistory_test_comparison.csv"
    metrics_by_home_path = NO_HISTORY_DIR / "metrics_by_home.csv"

    joblib.dump(model, model_path)
    joblib.dump(preprocessor, preprocessor_path)

    prediction_benchmark = benchmark_prediction_speed(model, X_test)

    feature_config = {
        "mode": "no_history",
        "model_family": "RandomForestRegressor",
        "model_name": "plegma_rf_no_history_simple_final_refit",
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "target_col": TARGET_COL,
        "time_col": TIME_COL,
        "home_col": HOME_COL,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN_NO_HISTORY,
        "uses_home_id_as_feature": False,
        "requires_user_home_id": False,
        "uses_lag_features": False,
        "uses_rolling_features": False,
        "uses_home_stats": False,
        "uses_knn_profiles": False,
        "uses_behavior_profiles": False,
        "default_prediction_column": "prediction_Wh",
    }
    save_json(feature_config, feature_config_path)

    test_eval.to_csv(predictions_path, index=False)
    test_eval.rename(columns={"prediction_Wh": "rf_no_history_prediction_Wh"}).to_csv(comparison_path, index=False)
    metrics_by_home(test_eval, "prediction_Wh").to_csv(metrics_by_home_path, index=False)

    metadata = {
        "model_family": "RandomForestRegressor",
        "model_name": "plegma_rf_no_history_simple_final_refit",
        "mode": "no_history",
        "rf_params": RF_NO_HISTORY_PARAMS,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN_NO_HISTORY,
        "clip_bounds": clip_bounds,
        "training_seconds": train_seconds,
        "model_file_size_mb": file_size_mb(model_path),
        "preprocessor_file_size_mb": file_size_mb(preprocessor_path),
        "prediction_benchmark": prediction_benchmark,
        "original_train_rows": int(len(train_df)),
        "final_train_rows_including_val_cal": int(len(final_train_df)),
        "used_train_rows": int(len(train_df_model)),
        "metrics_calibration_diagnostic": metrics_cal,
        "metrics_test": metrics_test,
        "artifact_files": {
            "model": str(model_path),
            "preprocessor": str(preprocessor_path),
            "feature_config": str(feature_config_path),
            "metadata": str(metadata_path),
            "predictions": str(predictions_path),
            "comparison": str(comparison_path),
            "metrics_by_home": str(metrics_by_home_path),
        },
        "train_period": {"start": str(train_df[TIME_COL].min()), "end": str(train_df[TIME_COL].max()), "rows": int(len(train_df))},
        "early_stop_validation_period_not_used_by_rf": {"start": str(es_df[TIME_COL].min()), "end": str(es_df[TIME_COL].max()), "rows": int(len(es_df))},
        "calibration_validation_period": {"start": str(cal_df[TIME_COL].min()), "end": str(cal_df[TIME_COL].max()), "rows": int(len(cal_df))},
        "final_train_period_used": {"start": str(final_train_df[TIME_COL].min()), "end": str(final_train_df[TIME_COL].max()), "rows": int(len(final_train_df))},
        "test_period": {"start": str(test_df[TIME_COL].min()), "end": str(test_df[TIME_COL].max()), "rows": int(len(test_df))},
    }
    save_json(metadata, metadata_path)

    return {
        "metrics_calibration_diagnostic": metrics_cal,
        "metrics_test": metrics_test,
        "model_path": str(model_path),
        "preprocessor_path": str(preprocessor_path),
        "feature_config_path": str(feature_config_path),
        "metadata_path": str(metadata_path),
        "predictions_path": str(predictions_path),
        "comparison_path": str(comparison_path),
        "metrics_by_home_path": str(metrics_by_home_path),
        "training_seconds": train_seconds,
        "model_file_size_mb": file_size_mb(model_path),
        "prediction_benchmark": prediction_benchmark,
    }


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    print_block("PLEGMA_RF_FINAL_GENERIC_NO_HOME_ID_V2")
    print("FINAL choices: with_history_generic = RF tuned params without home_id/stats/regime | no_history = nh_regularized_leaf20_depth18")
    print("with_history policy: home_id OFF as feature | home_stats OFF | consumption_regime OFF")
    print("home_id is used only internally for lag construction/evaluation grouping")
    print("no_history policy: home_id OFF | history OFF | behavior profiles OFF | KNN OFF")
    print("criterion: squared_error only | absolute_error OFF")

    df_base, removed_bad_homes = load_and_prepare_base_data()

    results: Dict[str, Any] = {
        "data_path": str(DATA_PATH),
        "out_root": str(OUT_ROOT),
        "time_col": TIME_COL,
        "home_col": HOME_COL,
        "target_col": TARGET_COL,
        "test_days": TEST_DAYS,
        "val_days_total": VAL_DAYS_TOTAL,
        "cal_days": CAL_DAYS,
        "use_log_target": USE_LOG_TARGET,
        "removed_bad_homes": removed_bad_homes,
        "num_rows_after_loading": int(len(df_base)),
        "num_homes_after_loading": int(df_base[HOME_COL].nunique()),
        "date_range": {
            "start": str(df_base[TIME_COL].min()),
            "end": str(df_base[TIME_COL].max()),
        },
    }

    if TRAIN_WITH_HISTORY:
        results["with_history"] = train_with_history_model(df_base)

    if TRAIN_NO_HISTORY:
        results["no_history"] = train_no_history_model(df_base)

    summary_path = OUT_ROOT / "training_summary.json"
    save_json(results, summary_path)

    print_block("FINAL PLEGMA API RF MODELS TRAINING COMPLETED")
    print("OUT_ROOT:", OUT_ROOT)
    print("With_history folder:", WITH_HISTORY_DIR)
    print("No_history folder:", NO_HISTORY_DIR)
    print("Training summary:", summary_path)

    if "with_history" in results:
        print("With_history RF model:", results["with_history"]["model_path"])
        print("With_history RF size MB:", results["with_history"]["model_file_size_mb"])

    if "no_history" in results:
        print("No_history RF model:", results["no_history"]["model_path"])
        print("No_history RF size MB:", results["no_history"]["model_file_size_mb"])


if __name__ == "__main__":
    main()


PLEGMA_RF_FINAL_GENERIC_NO_HOME_ID_V2
FINAL choices: with_history_generic = RF tuned params without home_id/stats/regime | no_history = nh_regularized_leaf20_depth18
with_history policy: home_id OFF as feature | home_stats OFF | consumption_regime OFF
home_id is used only internally for lag construction/evaluation grouping
no_history policy: home_id OFF | history OFF | behavior profiles OFF | KNN OFF
criterion: squared_error only | absolute_error OFF
Loading PLEGMA dataset
Rows loaded: 71,111
Homes loaded: 10
Date range: 2022-07-11 14:00:00 to 2023-10-01 01:00:00
WITH_HISTORY GENERIC TRAINING - FINAL PLEGMA RF
Rows after history features: 67,751
Homes after history features: 10
With_history generic temporal split
Train: 2022-07-25 14:00:00 to 2023-08-02 01:00:00 | rows: 54,795
Early-stop val: 2023-08-02 02:00:00 to 2023-08-17 01:00:00 | rows: 3,240
Calibration val: 2023-08-17 02:00:00 to 2023-09-01 01:00:00 | rows: 3,240
Test: 2023-09-01 02:00:00 to 2023-10-01 01:00:00 | rows: 6,476
Wi

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    7.6s
[Parallel(n_jobs=-1)]: Done 152 tasks      | elapsed:   56.6s
[Parallel(n_jobs=-1)]: Done 402 tasks      | elapsed:  2.4min
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:  2.9min finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:    0.1s
[Parallel(n_jobs=24)]: Done 500 out of 500 | elapsed:    0.2s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:    0.1s
[Parallel(n_jobs=24)]: Done 500 out of 500 | elapsed: 

DIAGNOSTIC METRICS ON CALIBRATION PERIOD - RF WITH_HISTORY GENERIC
MAE: 206.9795
RMSE: 443.2848
MAPE_%: 47.3620
sMAPE_%: 38.7848
bias_Wh: -107.3782
daily_abs_error_kWh_mean: 2.7118
daily_abs_error_pct_mean: 25.2491
num_home_days: 144
FINAL TEST METRICS - RF WITH_HISTORY GENERIC
MAE: 166.9575
RMSE: 395.8633
MAPE_%: 48.7450
sMAPE_%: 40.0786
bias_Wh: -69.8326
daily_abs_error_kWh_mean: 1.7766
daily_abs_error_pct_mean: 22.5647
num_home_days: 279


[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 500 out of 500 | elapsed:    0.0s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 500 out of 500 | elapsed:    0.1s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:    0.1s
[Parallel(n_jobs=24)]: Done 500 out of 500 | elapsed: 

NO_HISTORY TRAINING - FINAL PLEGMA RF
No_history temporal split
Train: 2022-07-11 14:00:00 to 2023-08-02 01:00:00 | rows: 58,155
Early-stop val: 2023-08-02 02:00:00 to 2023-08-17 01:00:00 | rows: 3,240
Calibration val: 2023-08-17 02:00:00 to 2023-09-01 01:00:00 | rows: 3,240
Final train used: 2022-07-11 14:00:00 to 2023-09-01 01:00:00 | rows: 64,635
Test: 2023-09-01 02:00:00 to 2023-10-01 01:00:00 | rows: 6,476
No_history RF features
Numeric features: 23
Categorical features: 9
Numeric: ['hour', 'day_of_week', 'is_weekend', 'is_holiday', 'month', 'external_temperature', 'internal_temperature', 'external_humidity', 'internal_humidity', 'num_rooms', 'residents', 'num_adults', 'num_children', 'num_elderly', 'has_ac', 'has_fridge_freezer', 'has_dryer', 'has_washing_machine', 'has_dishwasher', 'has_microwave', 'has_electric_oven', 'has_electric_hob', 'solar_panels']
Categorical: ['building_type', 'build_era', 'season', 'income_band', 'heating_type', 'water_heater_type', 'homeowner_status', 

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.8s
[Parallel(n_jobs=-1)]: Done 152 tasks      | elapsed:    7.0s
[Parallel(n_jobs=-1)]: Done 402 tasks      | elapsed:   17.3s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:   21.1s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:    0.1s
[Parallel(n_jobs=24)]: Done 500 out of 500 | elapsed:    0.1s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:    0.1s
[Parallel(n_jobs=24)]: Done 500 out of 500 | elapsed: 

DIAGNOSTIC METRICS ON CALIBRATION PERIOD - RF NO_HISTORY
MAE: 208.4931
RMSE: 442.6130
MAPE_%: 47.1801
sMAPE_%: 39.2024
bias_Wh: -125.1822
daily_abs_error_kWh_mean: 3.0631
daily_abs_error_pct_mean: 25.3793
num_home_days: 144
FINAL TEST METRICS - RF NO_HISTORY
MAE: 190.2944
RMSE: 429.1114
MAPE_%: 55.2818
sMAPE_%: 49.1954
bias_Wh: -97.4376
daily_abs_error_kWh_mean: 2.6982
daily_abs_error_pct_mean: 35.8763
num_home_days: 279


[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:    0.1s
[Parallel(n_jobs=24)]: Done 500 out of 500 | elapsed:    0.1s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 500 out of 500 | elapsed:    0.1s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:    0.1s
[Parallel(n_jobs=24)]: Done 500 out of 500 | elapsed: 

FINAL PLEGMA API RF MODELS TRAINING COMPLETED
OUT_ROOT: C:\Plegma_Programming\processed\models\final_api_models\RF2
With_history folder: C:\Plegma_Programming\processed\models\final_api_models\RF2\with_history
No_history folder: C:\Plegma_Programming\processed\models\final_api_models\RF2\no_history
Training summary: C:\Plegma_Programming\processed\models\final_api_models\RF2\training_summary.json
With_history RF model: C:\Plegma_Programming\processed\models\final_api_models\RF2\with_history\model.joblib
With_history RF size MB: 162.4778299331665
No_history RF model: C:\Plegma_Programming\processed\models\final_api_models\RF2\no_history\model.joblib
No_history RF size MB: 102.10121250152588
